# X-Mens decidual vs fibroblast DEGs compared with oil-induced artificial decidualization

* **X-Mens:** our snRNA-seq, decidual (`dec`) vs endometrial fibroblast (`eFib`) nuclei from the 3.5 dpi and
  4.5 dpi (menstrual) animals, apoptotic decidual nuclei (`decAp`) excluded. Pseudobulk per animal x cell type,
  paired within animal (`~animal + condition`). `timepoint` identifies the animal; its 10x lanes (`sample`)
  are summed, so lanes are never treated as replicates.
* **AD:** Wang et al. 2020, oil-stimulated vs unstimulated uterine horn on day 8 of pseudopregnancy,
  whole-uterus bulk RNA-seq (Supplementary Table S3). The table lists DEGs only.
* **DEG cutoffs (X-Mens):** padj <= 0.01, |fold change| >= 2, mean CPM > 3 in either cell type,
  following the fold-change and FDR thresholds of Wang et al.

Genes called in both lists are classified as shared up, shared down or divergent (opposite direction).

**Outputs** (in `OUT_DIR`):
* `XMens_vs_AD_DEGs.xlsx` - one workbook, one tab per table: summary, pseudobulks, X-Mens all genes,
  X-Mens DEGs, AD DEGs with their X-Mens result, shared DEGs
* `AD_XMens_shared_stacked.svg` - stacked bar of the shared DEGs

## 1. Libraries

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import matplotlib.pyplot as plt

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

plt.rcParams["svg.fonttype"] = "none"      # keep text editable in the SVG
pd.set_option("display.width", 200)

## 2. Datasets

In [ ]:
BULK_DIR = Path("/directoryto/BulkRnaSeq")
WANG_DIR = BULK_DIR / "Wang2020"

adata = sc.read_h5ad("/directoryto/Cevrim_Xmens_only_fibdec_cells.h5ad")
print(f"X-Mens: {adata.n_obs:,} nuclei x {adata.n_vars:,} genes")
print(pd.crosstab(adata.obs["timepoint"].astype(str), adata.obs["cell_type"].astype(str)).to_string())


def read_wang(table_no):
    # Read a Wang et al. 2020 supplementary DEG table (title in row 1, header in row 3)
    path = sorted(WANG_DIR.glob(f"*Supplementary_Table_{table_no}.xlsx"))
    if not path:
        raise FileNotFoundError(f"No *Supplementary_Table_{table_no}.xlsx in {WANG_DIR}")
    d = pd.read_excel(path[0], header=2).iloc[:, :9]
    d.columns = ["gene_symbol", "gene_id", "transcript_id", "mean_treated", "mean_control",
                 "fold_change", "pvalue", "FDR", "description"]
    d = d.dropna(subset=["gene_id"])
    d["gene_id"] = d["gene_id"].astype(str).str.split(".").str[0]
    d["direction"] = np.where(d["fold_change"] > 1, "up", "down")
    return d.set_index("gene_id")


AD = read_wang(3)
print(f"\nAD (Table S3): {len(AD):,} DEGs (up {(AD.direction == 'up').sum():,}, down {(AD.direction == 'down').sum():,})")

## 3. Variables

`TIMEPOINTS` lists the animals used; both cell types come from each animal (paired design).

In [ ]:
TIMEPOINTS = ["3.5dpi", "4.5dpi_menstrual"]   # animals used
EXCLUDE_SUBTYPES = ["decAp"]                  # apoptotic decidual nuclei

SAMPLE_KEY = "timepoint"                      # animal ("sample" = 10x lanes, summed)
CELLTYPE_COL = "cell_type"
SUBTYPE_COL = "cell_subtype"
TEST, REF = "dec", "eFib"                     # log2FC > 0 = higher in decidual nuclei
COUNTS_LAYER = "raw_counts"
GENE_ID_COL = "gene_ids"                      # var column with Ensembl IDs
MIN_CELLS = 20                                # min nuclei per animal x cell type pseudobulk

# DEG cutoffs: padj <= 0.01, |FC| >= 2, mean CPM > 3 (fold-change and FDR thresholds of Wang et al. 2020)
FDR, FC, MIN_CPM = 0.01, 2.0, 3.0

N_CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count()))

OUT_DIR = Path("/n/lab_storage/mckinley_lab/Lab/Cagri/X-MensFigures/20261113_PaperFigures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MARKERS = ["Prl8a2", "Alpl", "Bmp2", "Wnt4", "Hand2", "Ptgs2", "Hoxa10", "Foxo1", "Pgr", "Esr1"]

# Check names and select nuclei
tp = adata.obs[SAMPLE_KEY].astype(str)
ct = adata.obs[CELLTYPE_COL].astype(str)
st = adata.obs[SUBTYPE_COL].astype(str)
unknown = set(TIMEPOINTS) - set(tp.unique())
assert not unknown, f"Not found in obs['{SAMPLE_KEY}']: {unknown}. Available: {sorted(tp.unique())}"
unknown = set(EXCLUDE_SUBTYPES) - set(st.unique())
assert not unknown, f"Not found in obs['{SUBTYPE_COL}']: {unknown}. Available: {sorted(st.unique())}"
assert len(TIMEPOINTS) >= 2, "Need at least 2 animals for a paired comparison"

keep = tp.isin(TIMEPOINTS).values & ct.isin([TEST, REF]).values & ~st.isin(EXCLUDE_SUBTYPES).values
sub = adata[keep]

print(f"Animals: {TIMEPOINTS}; excluded subtypes: {EXCLUDE_SUBTYPES}; {sub.n_obs:,} nuclei kept")
print("\nSubtypes kept (cell type x subtype):")
print(pd.crosstab(sub.obs[CELLTYPE_COL].astype(str), sub.obs[SUBTYPE_COL].astype(str)).to_string())
print("\nNuclei used (animal x cell type):")
print(pd.crosstab(sub.obs[SAMPLE_KEY].astype(str), sub.obs[CELLTYPE_COL].astype(str)).to_string())

## 4. Build pseudobulk counts (sum of raw counts per animal x cell type)

In [ ]:
X = sp.csr_matrix(sub.layers[COUNTS_LAYER])
gene_ids = pd.Index(sub.var[GENE_ID_COL].astype(str)).str.split(".").str[0]
sym_map = pd.Series(sub.var_names.astype(str), index=gene_ids)
sym_map = sym_map[~sym_map.index.duplicated()]

obs = pd.DataFrame({"animal": sub.obs[SAMPLE_KEY].astype(str).values,
                    "condition": sub.obs[CELLTYPE_COL].astype(str).values})
obs["pb_id"] = obs["animal"] + "__" + obs["condition"]
codes, pb_ids = pd.factorize(obs["pb_id"])
onehot = sp.csr_matrix((np.ones(len(codes)), (codes, np.arange(len(codes)))), shape=(len(pb_ids), len(codes)))
pb = pd.DataFrame((onehot @ X).toarray(), index=pb_ids, columns=gene_ids)
pb = pb.T.groupby(level=0).sum().T                     # merge duplicated Ensembl IDs

meta = obs.groupby("pb_id").agg(animal=("animal", "first"), condition=("condition", "first"),
                                n_cells=("animal", "size")).loc[pb.index]
small = meta.n_cells < MIN_CELLS
if small.any():
    print(f"Dropped {small.sum()} pseudobulks with < {MIN_CELLS} nuclei: {list(meta.index[small])}")
meta, pb = meta[~small], pb[~small]

# keep animals that have both cell types (paired design)
both = meta.groupby("animal")["condition"].nunique()
meta = meta[meta.animal.isin(both[both == 2].index)]
pb = pb.loc[meta.index]
assert meta.animal.nunique() >= 2, "Fewer than 2 animals with both cell types - cannot run a paired comparison"

print(f"{len(meta)} pseudobulks from {meta.animal.nunique()} animals; design: ~animal + condition")
print(meta[["animal", "condition", "n_cells"]].to_string())

## 5. PyDESeq2: X-Mens dec vs eFib

In [ ]:
counts = pb.round().astype(int)
counts = counts.loc[:, (counts >= 10).sum(axis=0) >= 2]
m = meta[["animal", "condition"]].copy()
m["condition"] = pd.Categorical(m["condition"], categories=[REF, TEST])
m["animal"] = m["animal"].astype("category")

inference = DefaultInference(n_cpus=N_CPUS)
dds = DeseqDataSet(counts=counts, metadata=m, design="~animal + condition", refit_cooks=True,
                   inference=inference, quiet=True)
dds.deseq2()
ds = DeseqStats(dds, contrast=["condition", TEST, REF], inference=inference, quiet=True)
ds.summary()
res = ds.results_df.copy()

cpm = pb.div(pb.sum(axis=1), axis=0) * 1e6
for lvl in (REF, TEST):
    res[f"meanCPM_{lvl}"] = cpm.loc[meta.index[meta.condition == lvl], res.index].mean()
expressed = res[[f"meanCPM_{REF}", f"meanCPM_{TEST}"]].max(axis=1) > MIN_CPM

sig = (res["padj"] <= FDR) & expressed
res["direction"] = "ns"
res.loc[sig & (res["log2FoldChange"] >= np.log2(FC)), "direction"] = "up"
res.loc[sig & (res["log2FoldChange"] <= -np.log2(FC)), "direction"] = "down"
res.insert(0, "gene_symbol", sym_map.reindex(res.index).values)
res.index.name = "gene_id"
res = res.sort_values("padj")

print(f"X-Mens {TEST} vs {REF} ({', '.join(TIMEPOINTS)}): {len(res):,} genes tested; "
      f"up {(res.direction == 'up').sum():,}, down {(res.direction == 'down').sum():,} "
      f"(padj <= {FDR}, |FC| >= {FC}, CPM > {MIN_CPM})")
print(res[res.gene_symbol.isin(MARKERS)][["gene_symbol", "log2FoldChange", "padj", "direction"]].round(4).to_string())

## 6. Compare with oil-induced artificial decidualization (AD)

Each AD DEG is matched to X-Mens by Ensembl ID, and by gene symbol for IDs not found in X-Mens
(Wang et al. used an older annotation). Genes called in both lists are shared up, shared down or divergent.

In [ ]:
# AD gene -> X-Mens gene: Ensembl ID first, then symbol
sym2id = res.reset_index().drop_duplicates("gene_symbol").set_index("gene_symbol")["gene_id"]
xid = pd.Series(AD.index.where(AD.index.isin(res.index)), index=AD.index)
by_id = xid.notna()
xid[~by_id] = sym2id.reindex(AD.loc[~by_id, "gene_symbol"].values).values
print(f"AD DEGs matched to X-Mens: {by_id.sum():,} by Ensembl ID, {xid[~by_id].notna().sum():,} by symbol, "
      f"{xid.isna().sum():,} not in X-Mens")

ad_table = AD[["gene_symbol", "direction", "fold_change", "FDR"]].rename(
    columns={"direction": "AD_direction", "fold_change": "AD_fold_change", "FDR": "AD_FDR"})
ad_table["XMens_gene_id"] = xid
r = res.reindex(xid.values)
ad_table["XMens_direction"] = r["direction"].fillna("not tested").values
ad_table["XMens_log2FC"] = r["log2FoldChange"].values
ad_table["XMens_padj"] = r["padj"].values
ad_table.index.name = "gene_id"

# Shared DEGs (one row per X-Mens gene)
shared = ad_table[ad_table.XMens_direction.isin(["up", "down"])].drop_duplicates("XMens_gene_id").copy()
shared["category"] = np.select(
    [(shared.AD_direction == "up") & (shared.XMens_direction == "up"),
     (shared.AD_direction == "down") & (shared.XMens_direction == "down")],
    ["Shared up", "Shared down"], default="Divergent")
shared = shared.sort_values(["category", "XMens_log2FC"], ascending=[True, False])

CATS = {"Shared up": "#c0392b", "Shared down": "#f1948a", "Divergent": "#4d4d4d"}
shared_counts = shared["category"].value_counts().reindex(list(CATS), fill_value=0)
print(f"\nAD DEGs: {len(AD):,}; X-Mens DEGs: {(res.direction != 'ns').sum():,}; shared: {len(shared):,}")
print(pd.DataFrame({"genes": shared_counts,
                    "% of shared": (100 * shared_counts / len(shared)).round(1)}).to_string())

## 7. Plot: shared DEGs between AD and X-Mens

In [ ]:
pct = 100 * shared_counts / shared_counts.sum()

fig, ax = plt.subplots(figsize=(3.6, 4.8))
bottom = 0.0
for cat, color in CATS.items():
    v, n = pct[cat], shared_counts[cat]
    ax.bar(0, v, bottom=bottom, color=color, width=0.6, label=cat)
    if v >= 4:
        ax.text(0, bottom + v / 2, f"{n:,}\n({v:.0f}%)", ha="center", va="center", fontsize=8,
                color="white" if color in ("#c0392b", "#4d4d4d") else "black")
    bottom += v
ax.set_xticks([0])
ax.set_xticklabels([f"AD and X-Mens\nn = {len(shared):,} shared DEGs"], fontsize=8)
ax.set_xlim(-0.6, 0.6)
ax.set_ylabel("% of shared DEGs")
ax.set_ylim(0, 100)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(OUT_DIR / "AD_XMens_shared_stacked.svg", bbox_inches="tight")
plt.show()
print(f"Saved {OUT_DIR / 'AD_XMens_shared_stacked.svg'}")

## 8. Save all tables in one Excel workbook

In [ ]:
summary = pd.DataFrame({
    "item": ["Animals", "Excluded subtypes", "Design", "DEG cutoffs",
             "X-Mens genes tested", "X-Mens DEGs up (higher in dec)", "X-Mens DEGs down (higher in eFib)",
             "AD DEGs (Wang et al. 2020, Table S3)", "Shared DEGs", "Shared up", "Shared down", "Divergent"],
    "value": [", ".join(TIMEPOINTS), ", ".join(EXCLUDE_SUBTYPES), "~animal + condition",
              f"padj <= {FDR}, |FC| >= {FC}, mean CPM > {MIN_CPM}",
              len(res), (res.direction == "up").sum(), (res.direction == "down").sum(),
              len(AD), len(shared), *[shared_counts[c] for c in CATS]]})

SHEETS = {
    "Summary": (summary, False),
    "Pseudobulks": (meta, True),
    "XMens_all_genes": (res, True),
    "XMens_DEGs": (res[res.direction != "ns"], True),
    "AD_DEGs_vs_XMens": (ad_table, True),
    "Shared_DEGs": (shared, True),
}
xlsx = OUT_DIR / "XMens_vs_AD_DEGs.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
    for sheet, (df, keep_index) in SHEETS.items():
        df.to_excel(writer, sheet_name=sheet, index=keep_index)
        print(f"  {sheet}: {len(df):,} rows")
print(f"Saved {xlsx}")